# Lab 2 Part 1: Revised Kaggle run

Import this one file into Kaggle and run it with a nominal 16-GB GPU and Internet enabled. All executable source is embedded; course PDFs download from pinned URLs with SHA-256 checks. The notebook creates its helper files automatically and uses an isolated pinned Python environment. No repository clone, extra Python upload or Hugging Face token is required for the public models. Its source is checked to stay below Kaggle's 1-MB saving limit.

This revision fixes incomplete table crops and row/column extraction, avoids attaching unrelated charts to text questions, preserves exact named images and uses natural JSON with strict validation rather than forcing every field token. Both baseline and final use Qwen2-VL-7B in NF4 with float16 vision. No competition reference answers or question-specific lookup rules are used by the answering pipeline. A higher leaderboard score is not guaranteed; compare the new measured results with your prior 0.419 run.

Only Part 1 runs here. Existing Part 2 results and Part 3 are not changed or rerun. On T4 x2 it uses one GPU, not pooled VRAM. The course requires the Part 1 inference notebook to finish within 90 minutes; elapsed time is recorded below, but download speed, hardware and quotas can affect it.

Attach the competition data if available. An attached Questions.csv is selected automatically; the embedded practice file is the fallback. Keep the notebook private. For a background run, use Save Version > Save & Run All. A new session cannot recover unsaved files from an old session automatically.

In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, subprocess, json, hashlib, base64, io, stat, zipfile
SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
"""Kaggle-specific secrets, temporary caches and persistent result delivery."""
import hashlib
import os
from pathlib import Path
import shutil
import zipfile


def configure_runtime(working=None, scratch=None, environ=None, secret_loader=None, smoke_only=False,
                      require_private_artifacts=True):
    """Configure before ML imports. Never prompt for a token during a batch run."""
    if smoke_only:
        return {"mode": "CPU smoke only; no Kaggle GPU execution is certified"}
    environ = os.environ if environ is None else environ
    working = Path(working or "/kaggle/working").resolve()
    scratch = Path(scratch or "/kaggle/temp/lab2-runtime").resolve()
    if not working.is_dir():
        raise RuntimeError("Import this notebook into Kaggle, select a GPU and enable Internet before running it.")
    if scratch == working or working in scratch.parents:
        raise ValueError("Large caches and environments must be outside Kaggle's saved output directory.")
    token = environ.get("HF_TOKEN", "").strip()
    if not token and require_private_artifacts:
        try:
            if secret_loader is None:
                from kaggle_secrets import UserSecretsClient
                secret_loader = UserSecretsClient().get_secret
            token = secret_loader("HF_TOKEN").strip()
        except Exception:
            raise RuntimeError("In Kaggle, open Add-ons > Secrets, add HF_TOKEN with your Hugging Face read token, "
                               "enable it for this notebook, then rerun. Never paste the token into a cell.") from None
    if not token and require_private_artifacts:
        raise RuntimeError("The HF_TOKEN secret is empty. Set a read token with access to your private Part 2 artifacts.")
    scratch.mkdir(parents=True, exist_ok=True)
    if token:
        environ["HF_TOKEN"] = token
    # A T4 x2 runtime still uses one card. Two devices are not one 32-GB GPU.
    visible = environ.get("CUDA_VISIBLE_DEVICES", "0") or "0"
    environ["CUDA_VISIBLE_DEVICES"] = visible.split(",")[0]
    paths = {
        "HF_HOME": scratch / "huggingface",
        "HF_HUB_CACHE": scratch / "huggingface/hub",
        "HF_DATASETS_CACHE": scratch / "datasets",
        "TORCH_HOME": scratch / "torch",
        "XDG_CACHE_HOME": scratch / "cache",
        "UV_CACHE_DIR": scratch / "uv-cache",
        "UV_PYTHON_INSTALL_DIR": scratch / "uv-python",
        "LAB2_ENV_DIR": scratch / "environments",
        "LAB2_BOOTSTRAP_DIR": scratch / "bootstrap",
    }
    for name, path in paths.items():
        environ[name] = str(path)
    environ["LAB2_KAGGLE_WORKING"] = str(working)
    environ["LAB2_STANDALONE_PARENT"] = str(working)
    # Return paths and hardware selection, never the secret.
    return {"working": str(working), "temporary_cache": str(scratch),
            "visible_gpu": environ["CUDA_VISIBLE_DEVICES"], "secret_loaded": bool(token)}


def publish_zip(archive, working=None, display_link=True):
    """Keep the exact validated archive at the top of Kaggle's Output panel."""
    archive = Path(archive).resolve()
    working = Path(working or "/kaggle/working").resolve()
    if not working.is_dir() or not archive.is_file() or archive.suffix.lower() != ".zip":
        raise FileNotFoundError("A complete result ZIP and an existing output directory are required.")
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise ValueError("Result ZIP failed its integrity check; it was not published.")
    def checksum(path):
        value = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(2**20), b""):
                value.update(block)
        return value.hexdigest()
    destination = working / archive.name
    expected = checksum(archive)
    if destination.is_symlink():
        raise FileExistsError("The output destination is a symlink and was preserved.")
    if destination != archive:
        if destination.exists():
            if checksum(destination) != expected:
                raise FileExistsError("A different same-named result ZIP exists and was preserved.")
        else:
            if shutil.disk_usage(working).free < archive.stat().st_size + 64 * 2**20:
                raise RuntimeError("Not enough space for the Output-panel copy. The original ZIP is preserved.")
            shutil.copy2(archive, destination)
    if checksum(destination) != expected:
        raise ValueError("Output copy checksum mismatch. Keep the original ZIP for recovery.")
    print("KAGGLE OUTPUT ZIP:", destination.name)
    print("SHA256:", expected)
    print("Download this file from Kaggle's Output panel. Save the completed version with outputs.")
    if display_link:
        from IPython.display import FileLink, display
        # Kaggle's normal notebook cwd is /kaggle/working. The Output panel is
        # the reliable fallback if a frontend does not render a file link.
        if Path.cwd().resolve() == working:
            display(FileLink(destination.name))
    return destination

print("Runtime setup:", configure_runtime(smoke_only=SMOKE_ONLY, require_private_artifacts=False))
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
if not SMOKE_ONLY:
    try:
        print(subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=15))
    except (OSError, subprocess.SubprocessError) as error:
        raise RuntimeError("Select a GPU in Kaggle Session options before running. Setup and model work have not started.") from error
BUNDLE_SHA256 = '1ffcd867a7b18d4c9d9f8fff71dd9eaca0d60e60b5a26ef31cfc3bd2d7767203'
BUNDLE_BASE64 = (
    '...'
)
def unpack(data, expected_sha256, parent):
    """No pre-existing repo, git executable, or package import is required."""
    if hashlib.sha256(data).hexdigest() != expected_sha256:
        raise ValueError("The embedded notebook bundle is incomplete or changed.")
    root = Path(parent) / ("Lab2_Standalone_" + expected_sha256[:16])
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        if archive.testzip() is not None:
            raise ValueError("The embedded ZIP failed its integrity check.")
        manifest = json.loads(archive.read("STANDALONE_MANIFEST.json"))
        files = manifest["files"]
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError("Duplicate archive paths are not permitted.")
        if set(archive.namelist()) != set(files) | {"STANDALONE_MANIFEST.json"}:
            raise ValueError("The archive does not match its file manifest.")
        # Check every path and hash before creating or changing any file.
        for member in archive.infolist():
            name = member.filename
            path = PurePosixPath(name)
            if path.is_absolute() or ".." in path.parts or "\\" in name or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Unsafe embedded path.")
            if member.is_dir():
                raise ValueError("Only regular manifest files are permitted.")
            if name != "STANDALONE_MANIFEST.json" and hashlib.sha256(archive.read(name)).hexdigest() != files[name]:
                raise ValueError(f"Embedded source hash mismatch: {name}")
            target = root / name
            if target.is_symlink() or not target.resolve().is_relative_to(root.resolve()):
                raise ValueError("An extraction target escaped the standalone folder.")
            if target.is_file() and target.read_bytes() != archive.read(name):
                raise ValueError(f"Local edits at {target} were preserved. Use a new empty parent folder.")
        root.mkdir(parents=True, exist_ok=True)
        for name in archive.namelist():
            target = root / name
            if not target.is_file():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    return root

parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", "/kaggle/working"))
ROOT = unpack(base64.b64decode(BUNDLE_BASE64), BUNDLE_SHA256, parent)
os.environ["LAB2_STANDALONE_ROOT"] = str(ROOT)
print("Verified embedded source:", BUNDLE_SHA256)
from pathlib import Path
import os, sys, subprocess, json, time

ROOT = Path(os.environ["LAB2_STANDALONE_ROOT"]).resolve()
SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["LAB2_DEVICE"] = "cpu" if SMOKE_ONLY else "cuda"

# This override exists only for a developer CPU smoke, never for a GPU run.
test_python = os.environ.get("LAB2_STANDALONE_TEST_PYTHON", "") if SMOKE_ONLY else ""
if test_python:
    PYTHON = Path(test_python).expanduser().absolute()
    if not PYTHON.is_file():
        raise FileNotFoundError("The explicit CPU test interpreter does not exist.")
    subprocess.run([str(PYTHON), str(ROOT / "tools/preflight.py")], cwd=ROOT, check=True)
    (ROOT / ".lab2_runtime_cpu.json").write_text(json.dumps({"python": str(PYTHON), "cuda_requested": False}))
else:
    subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                    "--device", "cpu" if SMOKE_ONLY else "cuda"], cwd=ROOT, check=True)
    marker = ROOT / (".lab2_runtime_cpu.json" if SMOKE_ONLY else ".lab2_runtime_cuda.json")
    PYTHON = Path(json.loads(marker.read_text())["python"])
SEED = 266
print("Mode:", "CPU SMOKE ONLY: model calls are skipped" if SMOKE_ONLY else "FULL MODEL RUN")
print("Working folder:", ROOT)
print("Pinned interpreter:", PYTHON)

def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    command = [str(PYTHON), "-u", str(ROOT / "tools/monitored_run.py"), "--record", str(record),
               "--", str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f"{Path(relative).name} stopped with exit {status}. Saved log: {log}. Run the final ZIP cell to retain all partial results.")

run_script("tools/standalone_colab.py", "resources")
os.environ["LAB2_RAG_PROFILE"] = "7b-nf4"
os.environ["LAB2_RAG_NATURAL_JSON"] = "1"


## Run and keep the executed experiment

The isolated child notebook saves after every cell, so its outputs go into the result ZIP. Validation and checkpoint tests run before model generation. Errors are retained and partial results are exported automatically when possible. Unchanged completed answers can resume while this exact run folder still exists.

In [ ]:
run_script("tools/part1_kaggle.py", "run", *(["--smoke-only"] if SMOKE_ONLY else []))
print((ROOT / "runtime_checks/part1_completion.json").read_text())

## View measured results

No placeholder quality numbers are shown. If running a CPU smoke, model inference and quality measurements are skipped.

In [ ]:
if SMOKE_ONLY:
    print("CPU SMOKE ONLY. No leaderboard CSV or model-quality scores were created.")
else:
    from IPython.display import display, HTML
    import html
    metrics = json.loads((ROOT / "part1_rag/outputs/v2/evaluation/evaluation.json").read_text())
    columns = ["system", "correct_out_of_25", "retrieval_recall_at_5", "model_judged_citation_accuracy", "median_seconds_per_question"]
    header = "".join("<th>" + html.escape(key) + "</th>" for key in columns)
    rows = "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(key))) + "</td>" for key in columns) + "</tr>" for row in metrics["results"])
    display(HTML("<table><tr>" + header + "</tr>" + rows + "</table>"))
    print(metrics["limitations"])
    completion = json.loads((ROOT / "runtime_checks/part1_completion.json").read_text())
    print("Experiment elapsed minutes:", round(completion["seconds"] / 60, 2))
    print("Upload THIS new CSV, not the previous 0.419 file:", completion.get("competition_csv"))
    print("Within 90-minute experiment budget:", completion["seconds"] <= 5400)
    print("This timing excludes the first setup/install cell. Save Version's runtime is the whole-run timing.")

## Download the CSV, executed notebook and evidence

In Kaggle's Output panel download submission_part1_v2.csv (or its bundle-suffixed name if an older CSV was preserved), the Part 1 Results ZIP, and this notebook with its outputs. The exact new CSV path prints above. Do not accidentally resubmit the old 0.419 CSV. The ZIP includes the executed experiment, answer traces, evaluation, blank human-rating forms and diagnostic logs. It does not contain pretrained model caches or secrets. Running this cell never repeats model inference.

The completed run still needs real human ratings, team reference review, report/team records, a final tag and the actual Kaggle/Canvas submission. A green computation check does not stand in for those items.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
if "ROOT" not in globals():
    parent = Path(os.environ.get("LAB2_STANDALONE_PARENT", "/kaggle/working"))
    ROOT = parent / "Lab2_Standalone_1ffcd867a7b18d4c"
    if not (ROOT / "tools/part1_kaggle.py").is_file():
        raise FileNotFoundError("This session has no matching saved run. Open the saved version's Output panel; do not restart model work just to export.")
subprocess.run([sys.executable, str(ROOT / "tools/part1_kaggle.py"), "export"], cwd=ROOT, check=True)
print("Download the Lab2_Part1_Results ZIP from Output > /kaggle/working.")